# Create the inference wrapper

In order to use the model, we need to write a `PythonComponent` wrapper script that loads the model and feeds it by extracting the image from an `ImageSet` input payload, and prepares the output payload from the result of the segmentation. The `process_input` method will serve as an entrypoint for the wrapper.

An example script is already implemented under the `src` folder and in the `segmentation.py` file ([see here](../src/segmentation.py)). This notebook is purely for demonstrating how to write such a wrapper. Execution of this notebook is not expected, and changes being made in this tutorial will not be reflected in the final package.

## Load the model

The wrapper script must load the model first.

In [ ]:
from ultralytics import YOLO

model = YOLO(f"../models/yolo11n-seg.pt")

## Create ImageSet payload format

AI SDK provides an `ImageSet` class which makes it easier to work with this format. 
To be able to use it in your python component, the `imageset.py` file must be added to the pipeline components source folder, which can be achieved with the following code:

In [ ]:
import sys
from simaticai.common.resources import copy_resource_to
copy_resource_to('ImageSet', '../src')

Two new files have been created in the `../src` folder.
- [imageset.py](../src/imageset.py) which contains the methods for processing
- [requirements.txt](../src/requirements.txt) which contains the necessary dependencies

**&#9888; Note** Please make sure that the contents of [../src/requirements.txt](../src/requirements.txt) and the dependency list file you intend to use (in this demo, either requirements-cpu.txt or requirements-gpu.txt) are merged and all dependencies are installed before continuing.

Since this notebook also depends on `imageset.py`, we add the `src` folder to the path.

In [ ]:
sys.path.append('../src')
from imageset import ImageSet, ImageDetails, ImageFormat

The code below creates the required format, and puts the payload with the name `vision_payload` in the input dictionary. A similar dictionary will be provided by the AI Inference Server when an image arrives from the selected camera through the `Vision Connector Application`.

In [ ]:
image_detail = ImageDetails.from_image('../images/bus.jpg')

image_set = ImageSet(
    cameraid="camera_uuid",
    timestamp=image_detail.timestamp,
    detail=[image_detail]
)

input_payload = {'vision_payload': image_set.to_dict()}
input_payload

Examine the resulting dictionary. In case you don't want to use the `ImageSet` class provided by `AI SDK`. For further details, please refer to the [AI Inference Server Function Manual](https://docs.industrial-operations-x.siemens.cloud/r/en-us/2.6.0/ai-inference-server-function-manual/technical-information/data-acquisition-from-the-ie-vision-connector).

## Extract image from ImageSet

The wrapper script has to extract the image data from the payload, and create a BGR image for the model to process.

The original image is packaged into the payload in raw byte form, so the script has to convert it into a numpy array of the right shape.

In [ ]:
image_set = ImageSet.from_dict(input_payload['vision_payload'])
image_detail = image_set.detail[0]
width = image_detail.width
height = image_detail.height
print(f"Original image width: {width}, height: {height}")
image_data = image_set.get_image_rgb(0)  # RGB (height, width, 3)
print(f"Image data shape: {image_data.shape}")
print(f"Image data type: {image_data.dtype}")

## Inferencing the model

In [ ]:
result = model(image_data)
result = result[0]  # Get the first (and only) result from the list

## Postprocessing

The end goal of the wrapper is to create an output payload with the all the information we need. We plan to include the following data in the result:

- the id of the image (its path and name),
- the detected classes with their calculated relative areas, as shown in notebook [10-UltralyticsYoloModel](./10-UltralyticsYoloModel.ipynb),
- the annotated summary image to visualize (if visualization is requested).

The id of the image is simply acquired from the input payload.

In [ ]:
output_payload = {}
output_payload["iuid"] = image_detail.id

output_payload

Let's create a helper function that gathers all area-related information from a segmentation result into a dictionary.

In [ ]:
def calculate_areas(result):
    areas = []

    class_ids = result.boxes.cls.int().tolist()
    masks = result.masks.data

    for class_id, mask in zip(class_ids, masks):
        class_label = result.names[class_id]
        mask_area = mask.sum().item()
        total_area = mask.numel()
        relative_area = mask_area / total_area
        areas.append({"class": class_label,
                      "relative_area": relative_area,
                      "text": f"Detected {class_label} occupying {relative_area * 100.0:.1f}% of the image."})
    return areas

calculate_areas(result)

If we want to add a dictionary to the output payload, we have to convert it into a `json` string. 

In [ ]:
import json
output_payload["areas"] = json.dumps(calculate_areas(result))

Finally, we want to put the annotated image provided by the YOLO model into the output payload, so that we can inspect it on the AI Inference Server. However, we only want to do this if visualization from the AI IS side is specifically requested.

When visualization is required, AI IS updates a boolean parameter called `__AI_IS_IMAGE_SET_VISUALIZATION` on the wrapper. We have to prepare out script by implementing the `update_parameters` function.

In [ ]:
__AI_IS_IMAGE_SET_VISUALIZATION = False  # By default we do not visualize the annotated image

def update_parameters(parameters: dict):
    global __AI_IS_IMAGE_SET_VISUALIZATION 
    __AI_IS_IMAGE_SET_VISUALIZATION = parameters.get("__AI_IS_IMAGE_SET_VISUALIZATION", __AI_IS_IMAGE_SET_VISUALIZATION)

If the visualization parameter is set to true, we put the annotated image into the output payload in an ImageSet format. To do that, we reuse the ImageSet of the input payload, and replace the raw image data with the annotated image.

In [ ]:
update_parameters({"__AI_IS_IMAGE_SET_VISUALIZATION": True})  # Enable visualization; this is called by AI IS when visualization is requested

if __AI_IS_IMAGE_SET_VISUALIZATION is True:
    result_img = result.plot()
    # Replace the original image with the result image in ImageSet
    image_set.detail[0].update_image(result_img, ImageFormat.RGB8)
    # image_set['detail'][0]['image'] = result_img.ravel().tobytes()
    output_payload["result_image_set"] = image_set.to_dict()

output_payload

With that, our output payload is ready. You can find the entire code for the wrapper in the [segmentation.py](../src/segmentation.py) code.

To make the script easier to read and to make the tutorial more general, we created an additional script, [imageset.py](../src/imageset.py), which contains a class `ImageSet` that handles the general processing of ImageSet formats. The class can read and prepare vision payload dictionaries, change the stored images, query the images in RGB format, and so on.

We will include this file among the resources when we create the pipeline in notebook [30-CreatePipeline](./30-CreatePipeline.ipynb).

## Ultralytics unsupported dependency workaround

Unfortunately, Ultralytics has unsupported dependencies which cannot be used on AI Inference Server at the time of writing. These dependencies are `opencv-python`, which depends on libGL library and that is not presented on AI Inference Server. AI Inference Server version older than 2.9.0 also lacks the support of GPU-capable `pytorch`.

AI SDK can detect `pytorch` and automatically replaces it with a CPU only version if needed. However, replacing `opencv-python` with `opencv-python-headless`, which is the same package but without GUI toolkit dependencies, is not that easy.

Nonetheless, AI SDK provides a workaround. __Note that this only works with AI SDK version 2.6 and above, on AI Inference Server version 2.6 or above.__

First, we create a `requirements.txt` file with all the direct and transitive dependencies of Ultralytics, as well as every other packages and dependencies we need for our wrapper. One way to do this is to run a pip dry-run install command, and gather all the package names and versions pip would install for our wrapper. There, we swap the `opencv-python` dependency to `opencv-python-headless` of the same version.

Depending on our needs, the resulted file can either look like [requirements-cpu.txt](../src/requirements-cpu.txt), or [requirements-gpu.txt](../src/requirements-gpu.txt), depending on whenever we want to utilize CPU or GPU for inference, which we provided in the `src` folder.

### Setting the requirements for the wrapper

When we create our wrapper, as seen in the [next notebook](./30-CreatePipeline.ipynb), we set the package requirements by providing the requirements.txt with `no_deps=True` parameter:

```python
component.set_requirements("../src/requirements-cpu.txt", no_deps=True)
# component.set_requirements("../src/requirements-gpu.txt", no_deps=True)  # choose this for the GPU version
```

This will tell AI SDK and AI IS to download and install all the packages without any of their transitive dependencies; hence, Ultralytics will use the headless version of opencv-python.

__Keep in mind that you have to be careful with constructing the requirements.txt, as missing packages can lead to runtime errors.__

See notebook [30-CreatePipeline](./30-CreatePipeline.ipynb) for how to create the inference wrapper.